# Article 3 — When to Give Your Agent Subagents (and When Not To)

A subagent is a tool whose implementation is another agent loop, run in a fresh context window, that returns a summary instead of a transcript. Many agents can read; one agent writes.

This notebook builds on Article 2's `Agent` and adds `src/subagents/`. Requires `ANTHROPIC_API_KEY`.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
from dotenv import load_dotenv; load_dotenv('../.env')
from pathlib import Path
from src.orchestration.agent import Agent
from src.tools.registry import Registry
from src.subagents.workspace import workspace_tools
from src.subagents.ledger import TokenLedger, LedgeredMemory
from src.subagents.spawn import Subagent
from src.subagents.fanout import FanOut

CORPUS = Path('..').resolve() / 'src'

## 1. The explorer's tools are read-only

Search and read are where the tokens go. Look at what one read returns.

In [ ]:
tools = workspace_tools(CORPUS)
print(Registry(tools).describe())
print(tools[1](tools[1].Args(path='memory/memory.py'))[:800])

## 2. Least privilege, in code

A subagent may only hold read-only tools. Try to hand it a writer:

In [ ]:
from pydantic import BaseModel
from src.tools.base import Tool

class WriteFile(Tool):
    name, description = 'write_file', 'Write a file.'
    class Args(BaseModel):
        path: str
    def __call__(self, args): return 'wrote'

try:
    Subagent('explore', 'd', tools + [WriteFile()])
except ValueError as e:
    print(e)

## 3. Run (a): the agent reads the repo itself

In [ ]:
QUESTION = ("How does this repo's agent keep its context under budget? Name the module and function, "
            "and say what happens to the oldest transcript items when the budget is full.")
CRITERIA = 'names the file and function, explains eviction, cites file paths'

ledger_a = TokenLedger()
agent = Agent(Registry(tools), max_steps=12, memory=LedgeredMemory('parent', ledger_a, budget=16_000))
print(agent.run(QUESTION, CRITERIA, ask_user=lambda q: 'use your judgement'))
print(ledger_a.report())

## 4. Run (b): the agent delegates reading to a subagent

The parent holds one tool. Every file the explorer reads stays in the explorer's window; only the `Summary` comes back.

In [ ]:
ledger_b = TokenLedger()
explorer = Subagent('explore', 'Delegate a read-only question about the repo; get back a summary with file pointers.',
                    tools=workspace_tools(CORPUS), ledger=ledger_b)
agent = Agent(Registry([explorer]), max_steps=12, memory=LedgeredMemory('parent', ledger_b, budget=16_000))
print(agent.run(QUESTION, CRITERIA, ask_user=lambda q: 'use your judgement'))
print(ledger_b.report())

## 5. Run (c): parallel fan-out over independent questions

In [ ]:
import time
ledger_c = TokenLedger()
ex = Subagent('explore', 'Delegate a read-only question about the repo.', tools=workspace_tools(CORPUS), ledger=ledger_c)
agent = Agent(Registry([ex, FanOut(ex, max_parallel=3)]), max_steps=12, memory=LedgeredMemory('parent', ledger_c, budget=16_000))
t0 = time.perf_counter()
print(agent.run('Answer three independent questions about this repo, citing file paths: (1) how working memory stays under its token budget; (2) which tools the guardrails deny outright; (3) which model the evaluation judge uses by default.', 'answers all three with file paths', ask_user=lambda q: 'use your judgement'))
print(f'{time.perf_counter() - t0:.1f}s')
print(ledger_c.report())

## 6. Read the ledgers side by side

The parent's peak context should fall from (a) to (b). The TOTAL should rise. Isolation moves tokens; it does not remove them.

In [ ]:
for name, l in [('a alone', ledger_a), ('b explorer', ledger_b), ('c fan-out', ledger_c)]:
    p = l.by_agent['parent']
    print(f'{name:<12} parent peak {p.peak_context:>7,}   all agents {l.total():>8,}')